In [ ]:
%load_ext autoreload
%autoreload 2
%cd -q ..
import pickle

import numpy as np
import os
import sys
import pickle
import random
import logging
import importlib
import pandas as pd
from copy import deepcopy
from math import factorial, sqrt
from types import SimpleNamespace
from typing import Dict, List
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator, MultipleLocator
from deap import creator, base, tools
from scipy.stats import mannwhitneyu

from impl import config as cfg
from impl.core.algorithm.base import Budget
from impl.core.algorithm.rnsga2 import selRNSGA2WithMemory
from impl.core.algorithm.rnsga3 import selRNSGA3WithMemory
from impl.core.mr.base_mr import Perturbations
from impl.ads.mr.predefined import mr_set1, mr_set3
from impl.ads.scenario.scenario_definition import ScenarioDefinition
from impl.ads.utils.math_utils import calculate_auc_improvements, area_under_curve, calculate_ds_improvements
from impl.ads.utils.visualization import visualize_archived_distinct_solutions, visualize_archived_solutions_by_gen, \
    visualize_archive_solution_over_generations, visualize_distinct_solution_over_simulations, \
    visualize_computational_efficiency, _filter_by_thresholds, style_map, verbose_map, default_style

# Workaround for scenario definition restructure
import impl.ads.scenario.scenario_definition as scen_def_module
import impl.ads.mr.mr as base_mr_module

sys.modules["impl.scenario.scenario_definition"] = scen_def_module
sys.modules["impl.mr.mr"] = base_mr_module

logger = logging.getLogger(__name__)

# Prepare Data

In [ ]:
exp_data = {
    'ccea-c': [
        '1748884695969-ccea-d240fc',
        '1748941748026-ccea-b9ac6b',
        '1748989829337-ccea-4ec741',
        '1749038770422-ccea-42ed4d',
        '1749777922892-ccea-2c25d2',
        '1749814642920-ccea-de1f87',
        '1749886823824-ccea-649444',
        '1749923655258-ccea-1ba284',
        '1751198102877-ccea-1addfe',
        '1751208800030-ccea-a4ce3b',
    ],
    
    'ccea': [
        '1750185618111-ccea-6e8d47',
        '1750198943587-ccea-7053fc',
        '1750206690539-ccea-5685b2',
        '1750216454876-ccea-7d20e2',
        '1750227193443-ccea-78d6d1',
        '1750239856908-ccea-022c28',
        '1750251392201-ccea-cf8492',
        '1750268611439-ccea-719936',
        '1750277989099-ccea-b24329',
    ],
    
    'ga': [
        '1748920884019-ga-b8096d',
        '1748973401212-ga-159d20',
        '1749020894373-ga-0c91c0',
        '1749070990957-ga-e5c457',
        '1749117177309-ga-e90e3a',
        '1749801622497-ga-1bf740',
        '1749865564502-ga-d4aeee',
        '1749911224188-ga-398954',
        '1749948108048-ga-cf2ad2',
    ],
    
    'rs': [
        '1748904187579-rs-eb1239',
        '1748957118058-rs-2e552d',
        '1749004788416-rs-6bf5f1',
        '1749054600895-rs-2ed24b',
        '1749101043616-rs-24f811',
        '1749789337635-rs-395f0a',
        '1749825433865-rs-6f42c1',
        '1749852996207-rs-73049d',
        '1749897744564-rs-122ef9',
        '1749934847144-rs-a62c00',
    ],
    
    'ccea-c+rt': [
        '1750434085129-ccea-ddb606',
        '1750449704726-ccea-5b378e',
        '1750463599493-ccea-3174c7',
        '1750474241344-ccea-de114c',
        '1750486589729-ccea-d7ee1e',
        '1750497161663-ccea-e0eee8',
        '1751223236951-ccea-ca103a',
        '1751231897173-ccea-a652fa',
        '1751243485332-ccea-36643e',
        '1751274772053-ccea-ac5042',
    ],
    
    'ccea-2.3-sc1+rt': [
        '1750785163861-ccea-21aab6',
        '1750803393939-ccea-e46291',
        '1750816760201-ccea-a491e1',
        '1750840993246-ccea-b56d46',
        '1750864129037-ccea-b6c460',
        '1750874763838-ccea-c37984',
        '1751036741158-ccea-8dbdc7',
        '1751052499693-ccea-37549d',
        '1751064557270-ccea-8d3741',
        '1751327671761-ccea-fa8901',
    ],
}

# Plots for RQ1

In [ ]:
set1_solution_stats = visualize_archived_distinct_solutions(
    {
        'ccea-c': exp_data['ccea-c'],
        'ccea': exp_data['ccea'],
        'ga': exp_data['ga'],
        'rs': exp_data['rs'],
        
    },
    fitness_thresholds=[0.5, 0.8, 1.0, 1.3, 1.5, 2],
    distance_thresholds=np.arange(0, 4.41, 0.4).round(1),
    mr_set=mr_set1, box=False, show=True)

In [ ]:
comparison = visualize_archived_distinct_solutions(
    {
        'ccea-c': exp_data['ccea-c'],
        'ccea': exp_data['ccea'],
        'ccea-c+ri': exp_data['ccea-c+rt'],
        'ccea+ri': exp_data['ccea+rt'],
        
    },
fitness_thresholds=[0.5, 0.8, 1.0, 1.3, 1.5, 2],
distance_thresholds=np.arange(0, 4.41, 0.4).round(1),
mr_set=mr_set1, box=False, show=True)

# Distance to Runtime Scenarios

In [ ]:
importlib.import_module("impl.problem")
runtime_scenarios = []
for data_path in cfg.CONFIG["workspace"]["runtime_scenario"].rglob("*.*"):
    runtime_scenarios += pickle.loads(data_path.read_bytes())
runtime_scenarios = [creator.Scenario(scenario) for scenario in runtime_scenarios]
runtime_scenarios = [scen for scen in runtime_scenarios if count_actors_scen(scen) > 1]
projects = {
    'ccea-c': exp_data['ccea-c'],
    'ccea': exp_data['ccea'],
    'ccea-c+ri': exp_data['ccea-c+rt'],
    'ccea+ri': exp_data['ccea+rt'],
    'ga': exp_data['ga'],
    'rs': exp_data['rs'],
}

def get_sims(_solutions):
    _sims = []
    for _source, _perturbation in _solutions:
        source_sim = min([_source.dist(scen) for scen in runtime_scenarios])
        _follow_up = deepcopy(_source)
        _follow_up.assign_new_id()
        _perturbation.perturb(_follow_up)
        follow_up_sim = min([_follow_up.dist(scen) for scen in runtime_scenarios])
        _sims.append(min(follow_up_sim, source_sim))
    return _sims

sims = {}
for alg, project_list in projects.items():
    solutions = []
    for project in project_list:
        sol_file = next((cfg.CONFIG["workspace"]["result"] / project / "solutions").rglob("solutions*"), None)
        if sol_file is None: continue
        solutions += pickle.loads(sol_file.read_bytes())
    proj_sim = get_sims(solutions)
    print(f"{alg} sim: {sum(proj_sim) / len(proj_sim)}")
    sims[alg] = proj_sim
    
fig, ax = plt.subplots(figsize=(10,6))
bplot = ax.boxplot(sims.values(), patch_artist=True, showmeans=True, whis=10,
                   labels=[verbose_map.get(alg,alg) for alg in sims.keys()])
cmap = cm.ScalarMappable(cmap="rainbow")
for patch, color in zip(bplot["boxes"], [style_map.get(alg, default_style)['color'] for alg in sims.keys()]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel("Distance to Runtime Scenarios")
ax.grid()
fig.tight_layout()
fig.savefig(cfg.CONFIG["workspace"]["visualization"] / "distance_to_runtime.png")
plt.show()

# Top Fitnesses

In [ ]:
fitnesses = {}
avg_fitnesses = {}
top_fitnesses = {}
for alg, project_list in projects.items():
    avg_fitnesses[alg] = []
    solutions = []
    top_solutions = []
    for project in project_list:
        sol_file = next((cfg.CONFIG["workspace"]["result"] / project / "solutions").rglob("solutions*"), None)
        if sol_file is None: continue
        run_sols = pickle.loads(sol_file.read_bytes())
        top_size = round(len(run_sols) / 5)
        run_top = sorted(run_sols, key=lambda sol: sol.fitness.values[0])[-top_size:] if top_size > 0 else []
        solutions += run_sols
        top_solutions += run_top
        avg_fitness = sum([sol.fitness.values[0] for sol in run_sols]) / len(run_sols)
        avg_fitnesses[alg].append(avg_fitness)
    proj_fitness = [sol.v2.fitness[0] - sol.v1.fitness[0] for sol in solutions]
    top_proj_fitness = [sol.fitness.values[0] for sol in top_solutions]
    fitnesses[alg] = proj_fitness
    top_fitnesses[alg] = top_proj_fitness
    
overall_top_fitnesses = {alg: sorted(fitnesses[alg])[-round(len(fitnesses[alg]) / 5):] for alg in fitnesses}
fig, ax = plt.subplots(figsize=(10,6))
bplot = ax.boxplot(overall_top_fitnesses.values(), patch_artist=True, showmeans=True, whis=1,
                   labels=[verbose_map.get(alg,alg) for alg in overall_top_fitnesses.keys()])
cmap = cm.ScalarMappable(cmap="rainbow")
for patch, color in zip(bplot["boxes"], cmap.to_rgba([np.mean(fit) for fit in overall_top_fitnesses.values()])):
    patch.set_facecolor(color)
    patch.set_alpha(0.4)
ax.set_ylabel("Difference of Violation")
ax.grid()
fig.tight_layout()
fig.savefig(cfg.CONFIG["workspace"]["visualization"] / "top_fitnesses.png")
plt.show()